# <span style="font-size: 19.2213px; white-space: pre;">HÁZI FELADAT</span>

## Case tábla

1. Case tábla Customer tábla alapján. 
2. caseID = customerID, 
3. case level attributumok: name, country, favourite\_product\_category (=legnagyobb értékben rendelte)

In [11]:
-- Case tábla

SELECT top(3)
    c.customer_id AS case_id,
    c.name,
    c.country,
    (
        SELECT TOP 1 p.category
        FROM sql_recap.dbo.orders o
        LEFT JOIN sql_recap.dbo.order_details i ON o.order_id = i.order_id
        LEFT JOIN sql_recap.dbo.products p ON i.product_id = p.product_id
        WHERE o.customer_id = c.customer_id
        GROUP BY p.category
        ORDER BY SUM(i.quantity * i.price_per_unit) DESC
    ) AS favourite_product_category
FROM sql_recap.dbo.customers c;

--a szükséges adatmezők
    --select c.customer_id as caseID, name, signup_date, country from sql_recap.dbo.customers c
    --select o.order_id, o.customer_id from sql_recap.dbo.orders o
    --select i.order_id, i.product_id, quantity*price_per_unit as 'item_sum' from sql_recap.dbo.order_details i
    --select p.product_id, p.category from sql_recap.dbo.products p

--sanity check
    --select o.customer_id, p.category, sum(quantity*price_per_unit) as 'item_sum' 
    --from sql_recap.dbo.order_details i
    --left join sql_recap.dbo.orders o
    --on i.order_id = o.order_id
    --left join sql_recap.dbo.products p
    --on i.product_id = p.product_id
    --group by o.customer_id, p.category 
    --having o.customer_id in (1,2)
    --order by customer_id ASC, item_sum DESC

(3 rows affected)

Total execution time: 00:00:00.039

case_id,name,country,favourite_product_category
1,Dr. Erin Preston DDS,France,Toys
2,Brad Castro,USA,Books
3,Mrs. Erica Smith,Germany,Toys


## Event táblák

kapcsolódó eventek létrehozása. 

1\. customer sign-up event

2\. create order event

In [16]:
--Event tábla nr 1
--Customer sign-up

-- select csak top 3 a rövidség kedvéért
select top(3) c.customer_id as case_id, 'customer sign-up' as event_type, signup_date as 'event_time' 
from [sql_recap].[dbo].[customers] c


(3 rows affected)

Total execution time: 00:00:00.006

case_id,event_type,event_time
1,customer sign-up,2020-06-26
2,customer sign-up,2023-03-08
3,customer sign-up,2020-06-22


In [15]:
--Event tábla nr 2
-- Create order

-- select csak top 3 a rövidség kedvéért
select top(3) c.customer_id as case_id, 'create order' as event_type, order_date as event_time
from  [sql_recap].[dbo].[customers] c
left join  [sql_recap].[dbo].[orders] o
on c.customer_id = o.customer_id

(3 rows affected)

Total execution time: 00:00:00.008

case_id,event_type,event_time
1,Create Order,2024-08-01
1,Create Order,2024-11-15
1,Create Order,2024-07-01


## Event log

In [27]:
-- event log letrehozas
select * from 

(
select c.customer_id as case_id, c.name, c.country, 'customer sign-up' as event_type, signup_date as 'event_time', 
(
        SELECT TOP 1 p.category
        FROM sql_recap.dbo.orders o
        LEFT JOIN sql_recap.dbo.order_details i ON o.order_id = i.order_id
        LEFT JOIN sql_recap.dbo.products p ON i.product_id = p.product_id
        WHERE o.customer_id = c.customer_id
        GROUP BY p.category
        ORDER BY SUM(i.quantity * i.price_per_unit) DESC
    ) AS favourite_product_category
FROM sql_recap.dbo.customers c

--
union all
--
Select c.customer_id as case_id, c.name, c.country, 'create order' as event_type, order_date as 'event_time',
(
        SELECT TOP 1 p.category
        FROM sql_recap.dbo.orders o
        LEFT JOIN sql_recap.dbo.order_details i ON o.order_id = i.order_id
        LEFT JOIN sql_recap.dbo.products p ON i.product_id = p.product_id
        WHERE o.customer_id = c.customer_id
        GROUP BY p.category
        ORDER BY SUM(i.quantity * i.price_per_unit) DESC
    ) AS favourite_product_category
from  [sql_recap].[dbo].[customers] c
left join  [sql_recap].[dbo].[orders] o
on c.customer_id = o.customer_id
) 

as event_log

order by case_id, event_time

-- sanity check
    --select c.customer_id as case_id, 'create order' as event_type, order_date as event_time
    --from  [sql_recap].[dbo].[customers] c
    --left join  [sql_recap].[dbo].[orders] o
    --on c.customer_id = o.customer_id
    --where c.customer_id in (1,2)
    --order by case_id, event_time

    --select c.customer_id as case_id, 'customer sign-up' as event_type, signup_date as 'event_time' 
    --from [sql_recap].[dbo].[customers] c
    --where c.customer_id in (1,2)


(3500 rows affected)

Total execution time: 00:00:01.581

case_id,name,country,event_type,event_time,favourite_product_category
1,Dr. Erin Preston DDS,France,customer sign-up,2020-06-26,Toys
1,Dr. Erin Preston DDS,France,create order,2024-02-19,Toys
1,Dr. Erin Preston DDS,France,create order,2024-02-27,Toys
1,Dr. Erin Preston DDS,France,create order,2024-03-02,Toys
1,Dr. Erin Preston DDS,France,create order,2024-07-01,Toys
1,Dr. Erin Preston DDS,France,create order,2024-08-01,Toys
1,Dr. Erin Preston DDS,France,create order,2024-08-03,Toys
1,Dr. Erin Preston DDS,France,create order,2024-11-15,Toys
1,Dr. Erin Preston DDS,France,create order,2024-11-15,Toys
2,Brad Castro,USA,customer sign-up,2023-03-08,Books
